# Chapter 2: Cash Flows, Discounting, and Present Value

This notebook accompanies the chapter in *Python & AI for Rates, Bonds, and Credit*. It focuses on reproducible inspection: run the chapter script, inspect the data snapshot, and connect the numerical result back to the fixed-income interpretation.

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SCRIPT = ROOT / "code" / "ch02_cash_flows_discounting.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

Use the script output as controlled evidence. Check the units, sign convention, source dataset, and the economic direction of the result before using it in a written explanation or GenAI-assisted summary.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch02_*.csv"))
summary = pd.DataFrame({
    "file": [p.name for p in chapter_files],
    "rows": [pd.read_csv(p).shape[0] for p in chapter_files],
    "columns": [pd.read_csv(p).shape[1] for p in chapter_files],
})
summary

## Chapter-specific checks

Rebuild the present-value contribution table and add a one-basis-point sensitivity check. The result should make both value contribution and rate sensitivity visible by payment date.

In [ ]:
import numpy as np

flows = pd.read_csv(DATA / "ch02_cashflows.csv")
rate = 0.04
flows["discount"] = np.exp(-rate * flows.time)
flows["pv_part"] = flows.cashflow * flows.discount
flows["pv_part_bumped"] = (
    flows.cashflow * np.exp(-(rate + 0.0001) * flows.time)
)
flows["dv01_part"] = flows.pv_part - flows.pv_part_bumped
flows.round(6)